# Florence2 Large Nocaps Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `NoCaps-Baseline/Florence-2.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 1. Uyumsuz olan dev sürümünü kaldır
!pip uninstall -y transformers

# 2. Florence-2 ile %100 uyumlu olan KARARLI sürümü kur
!pip install transformers==4.44.2
!pip install timm flash_attn einops accelerate

print("✅ Kurulum bitti. ŞİMDİ MUTLAKA RESTART YAPIN! 👇")

In [ ]:
import torch
import transformers
from transformers import AutoProcessor, AutoModelForCausalLM
from google.colab import drive
import os

# Versiyon Kontrolü (İçimiz rahat olsun)
print(f"Versiyon Kontrolü: {transformers.__version__}")
# Beklenen: 4.44.2

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

MODEL_ID = "microsoft/Florence-2-large"
print(f"⏳ {MODEL_ID} yükleniyor...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to("cuda").eval()

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("✅ BAŞARDIK! Model Hatasız Yüklendi.")

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from PIL import Image
import os
import json
from tqdm import tqdm
from google.colab import drive

# 1. Drive Bağlantısı (Garanti olsun)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Dosya Yolları (Hata Kontrollü)
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
IMG_DIR = "/content/drive/MyDrive/datasets/nocaps/images_val_hf"
OUTPUT_FILE = "florence2_nocaps_results.json"

if not os.path.exists(GT_PATH):
    raise FileNotFoundError(f"❌ JSON dosyası bulunamadı: {GT_PATH}\nLütfen Drive yolunu kontrol et.")

if not os.path.exists(IMG_DIR):
    raise FileNotFoundError(f"❌ Görsel klasörü bulunamadı: {IMG_DIR}")

# 3. Model Yükleme (Eğer hafızada yoksa yükler)
MODEL_ID = "microsoft/Florence-2-large"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"⏳ {MODEL_ID} yükleniyor...")
try:
    # Model zaten varsa tekrar yükleme (Zaman kazancı)
    if 'model' not in globals():
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            torch_dtype=torch.float16,
            trust_remote_code=True
        ).to(device).eval()
        print("✅ Model başarıyla yüklendi.")
    else:
        print("ℹ️ Model zaten hafızada, tekrar yüklenmiyor.")
except Exception as e:
    # Hata olursa sıfırdan yükle
    print(f"⚠️ Model yükleme hatası: {e}. Tekrar deneniyor...")
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        trust_remote_code=True
    ).to(device).eval()

# 4. Tahmin Döngüsü
with open(GT_PATH, "r", encoding="utf-8") as f:
    nocaps_gt = json.load(f)

results = []
TASK_PROMPT = "<CAPTION>"

print(f"🚀 {len(nocaps_gt['images'])} görsel işleniyor...")

for img_info in tqdm(nocaps_gt['images']):
    image_path = os.path.join(IMG_DIR, img_info['file_name'])

    if not os.path.exists(image_path):
        continue

    try:
        image = Image.open(image_path).convert("RGB")

        inputs = processor(text=TASK_PROMPT, images=image, return_tensors="pt").to(device, torch.float16)

        with torch.no_grad():
            generated_ids = model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=1024,
                num_beams=3,
                do_sample=False
            )

        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

        parsed_answer = processor.post_process_generation(
            generated_text,
            task=TASK_PROMPT,
            image_size=(image.width, image.height)
        )

        caption = parsed_answer[TASK_PROMPT]

        results.append({
            "image_id": img_info['id'],
            "caption": caption
        })

    except Exception as e:
        print(f"Hata ({img_info['file_name']}): {e}")

# 5. Kaydet
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\n💾 Sonuçlar kaydedildi: {OUTPUT_FILE}")

In [ ]:
import os
import sys
import json

# ==============================================================================
# 1. ORTAM HAZIRLIĞI
# ==============================================================================
print("🛠️ Değerlendirme ortamı hazırlanıyor...")

# Java Kurulumu (METEOR ve Tokenizer için şart)
os.system("apt-get install -y openjdk-8-jdk-headless -qq > /dev/null")
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# PyCocoEvalCap Kütüphanesi
if not os.path.exists("pycocoevalcap"):
    os.system("git clone https://github.com/salaniz/pycocoevalcap.git")

# Tokenizer Dosyasını Manuel İndir (Garanti Yöntem)
target_dir = "pycocoevalcap/tokenizer"
target_file = os.path.join(target_dir, "stanford-corenlp-3.4.1.jar")
os.makedirs(target_dir, exist_ok=True)

if not os.path.exists(target_file):
    print("⬇️ Tokenizer dosyası indiriliyor...")
    os.system(f"wget -q https://repo1.maven.org/maven2/edu/stanford/nlp/stanford-corenlp/3.4.1/stanford-corenlp-3.4.1.jar -O {target_file}")

# Kütüphane yolunu sisteme ekle
sys.path.append(os.path.abspath("pycocoevalcap"))

# ==============================================================================
# 2. HESAPLAMA (METEOR EKLENDİ)
# ==============================================================================
from tokenizer.ptbtokenizer import PTBTokenizer
from bleu.bleu import Bleu
from rouge.rouge import Rouge
from cider.cider import Cider
from meteor.meteor import Meteor # <-- METEOR eklendi

# Dosya Yolları
GT_PATH = "/content/drive/MyDrive/datasets/nocaps/nocaps_val_4500_captions_domain_norm.json"
RES_PATH = "florence2_nocaps_results.json"

if not os.path.exists(RES_PATH):
    raise FileNotFoundError("❌ Sonuç dosyası bulunamadı! Lütfen önce tahmin kodunu çalıştır.")

print(f"\n📊 Florence-2 Skorları Hesaplanıyor (METEOR Dahil)...")

coco = json.load(open(GT_PATH))
cocoRes = json.load(open(RES_PATH))

# Format Düzenleme
gts = {ann['image_id']: [] for ann in coco['annotations']}
for ann in coco['annotations']:
    gts[ann['image_id']].append(ann)

res = {ann['image_id']: [ann] for ann in cocoRes}

# Domain Grupları
ids_in   = [img['id'] for img in coco['images'] if img['domain_norm'] == 'in']
ids_near = [img['id'] for img in coco['images'] if img['domain_norm'] == 'near']
ids_out  = [img['id'] for img in coco['images'] if img['domain_norm'] == 'out']
ids_all  = [img['id'] for img in coco['images']]

def evaluate_manual(img_ids, title):
    if not img_ids: return
    print(f"\n{'='*10} {title} ({len(img_ids)}) {'='*10}")

    gts_curr = {i: gts[i] for i in img_ids if i in gts}
    res_curr = {i: res[i] for i in img_ids if i in res}

    # Tokenization
    tokenizer = PTBTokenizer()
    gts_tok = tokenizer.tokenize(gts_curr)
    res_tok = tokenizer.tokenize(res_curr)

    # Skorlama Listesi (METEOR Eklendi)
    scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Meteor(), "METEOR"),   # <-- Yeni eklenen metrik
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]

    for scorer, method in scorers:
        try:
            score, scores = scorer.compute_score(gts_tok, res_tok)
            if isinstance(method, list):
                for m, s in zip(method, score):
                    print(f"{m:10s}: {s:.3f}")
            else:
                print(f"{method:10s}: {score:.3f}")
        except Exception as e:
            print(f"⚠️ {method} hesaplanırken hata: {e}")

# Raporları Yazdır
evaluate_manual(ids_in, "IN-DOMAIN")
evaluate_manual(ids_near, "NEAR-DOMAIN")
evaluate_manual(ids_out, "OUT-OF-DOMAIN")
evaluate_manual(ids_all, "OVERALL (GENEL)")